# 0. Preperatrion

In [1]:
import datetime
import os
import pickle
import re

from tqdm import tqdm

from preprocess_5G import AmariNSALogFile
from dataloader import SrsRANLteHybridEncoder
import utils

In [2]:
all_segments = {
    64: {
        ("12:06:38", "12:10:03"): "live",
        ("12:10:04", "12:12:54"): "video",
        ("12:13:01", "12:15:58"): "meet",
        ("12:16:03", "12:18:46"): "call",
        ("12:18:50", "12:25:56"): "down",
        ("12:26:13", "12:29:16"): "up",
        ("12:33:52", "12:36:32"): "tiktok"
    },
    66: {
        ("21:00:16", "21:05:47"): "live",
        ("21:05:49", "21:10:18"): "video",
        ("21:10:46", "21:15:40"): "meet",
        ("21:15:46", "21:20:53"): "call",
        ("21:21:02", "21:23:32"): "down",
        ("21:24:01", "21:29:02"): "up",
        ("21:29:02", "21:35:08"): "tiktok"
    },
    68: {
        ("14:00:54", "14:05:20"): "live",
        ("14:05:43", "14:09:05"): "video",
        ("14:09:13", "14:14:21"): "meet",
        ("14:14:23", "14:17:45"): "call",
        ("14:17:53", "14:20:06"): "down",
        ("14:20:17", "14:23:20"): "up",
        ("14:24:15", "14:27:38"): "tiktok"
    },
    70: {
        ("17:45:03", "17:48:33"): "live",
        ("17:48:34", "17:50:41"): "video",
        ("17:51:19", "17:54:38"): "meet",
        ("17:54:39", "17:58:00"): "call",
        ("17:59:51", "18:00:36"): "down",
        ("18:01:03", "18:05:32"): "up",
        ("18:06:12", "18:10:04"): "tiktok"
    },
    72: {
        ("18:16:50", "18:23:01"): "live",
        ("18:23:21", "18:25:51"): "video",
        ("18:27:05", "18:29:37"): "meet",
        ("18:29:57", "18:33:03"): "call",
        ("18:34:18", "18:35:36"): "down",
        ("18:36:37", "18:41:43"): "up",
        ("18:43:02", "18:47:13"): "tiktok"
    },
    74: {
        ("12:02:00", "12:09:32"): "live",
        ("12:09:46", "12:16:42"): "video",
        ("12:17:30", "12:20:00"): "meet",
        ("12:20:01", "12:23:10"): "call",
        ("12:23:17", "12:25:41"): "down",
        ("12:26:03", "12:40:43"): "up",
        ("12:40:43", "12:42:53"): "tiktok",
    },
    76: {
        ("19:38:19", "19:41:34"): "live",
        ("19:42:30", "19:46:38"): "video",
        ("19:47:15", "19:50:48"): "meet",
        ("19:50:51", "19:53:49"): "call",
        ("19:54:01", "19:56:26"): "down",
        ("19:57:02", "20:04:04"): "up",
        ("20:05:01", "20:08:55"): "tiktok"
    },
    78: {
        ("18:50:13", "18:55:00"): "live",
        ("18:55:00", "18:59:12"): "video",
        ("18:59:22", "19:01:44"): "meet",
        ("19:01:46", "19:05:55"): "call",
        ("19:06:00", "19:08:26"): "down",
        ("19:09:02", "19:17:00"): "up",
        ("19:17:48", "19:20:55"): "tiktok"
    },
    80: {
        ("14:46:33", "14:52:37"): "live",
        ("14:52:56", "14:56:33"): "video",
        ("14:58:26", "15:01:02"): "meet",
        ("15:01:04", "15:06:07"): "call",
        ("15:06:12", "15:08:38"): "down",
        ("15:11:40", "15:22:19"): "up",
        ("15:24:03", "15:28:14"): "tiktok"
    },
    82: {
        ("20:11:05", "20:17:15"): "live",
        ("20:17:24", "20:22:10"):"video",
        ("20:22:10", "20:25:58"): "meet",
        ("20:26:04", "20:29:56"):"call",
        ("20:30:00", "20:34:09"): "down",
        ("20:35:02", "20:47:59"): "up",
        ("20:48:04", "20:51:57"):"tiktok"
    },
    84: {
        ("12:56:30", "12:59:29"): "live",
        ("12:59:43", "13:07:55"): "video",
        ("13:08:07", "13:11:07"): "meet",
        ("13:11:16", "13:14:24"): "call",
        ("13:14:33", "13:19:18"): "down",
        ("13:20:05", "13:35:14"): "up",
        ("13:35:33", "13:39:44"): "tiktok",
    },
}

In [3]:
gains =[64,66,68,70,72,74,76,78,80,82,84]

# 1. Remove the headers

In [ ]:
def remove_header(input_file_path, output_folder):
    """
    Reads a file and writes a new file excluding lines that start with '#'.
    The output file is named after the original file name appended with the output folder's name.

    Args:
    input_file_path (str): Path to the input file.
    output_folder (str): Folder where the output file will be saved.
    """
    # Extract the base name of the input file
    base_name = os.path.basename(input_file_path)
    # Create a new output file name by appending the folder name to the original file name
    new_file_name = f"{base_name}_{os.path.basename(output_folder)}.log"

    # Ensure the output directory exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    output_file_path = os.path.join(output_folder, new_file_name)

    try:
        with open(input_file_path, 'r') as file:
            lines = file.readlines()

        # Filter out lines that start with '#'
        filtered_lines = [line for line in lines if not line.strip().startswith('#')]

        # Write the filtered lines to the output file
        with open(output_file_path, 'w') as file:
            file.writelines(filtered_lines)

    except Exception as e:
        print(f"An error occurred: {e}")
        

for gain in gains: 
    # NOTE:
    # Replace the following absolute paths with your local dataset path.
    raw_dir = DATA_ROOT / "traffic-identification" / "NR" / "SA" / str(gain) / "raw"
    output_dir = DATA_ROOT / "traffic-identification" / "NR" / "SA" / str(gain) / "headless_fixed"
    output_dir.mkdir(parents=True, exist_ok=True)
    for filename in tqdm(os.listdir(raw_dir), postfix={"gain": gain}): 
        input_file_path = os.path.join(raw_dir, filename)
        remove_header(input_file_path, output_dir)

# 2. Concat headless

In [ ]:
def concatenate_files_in_directory(directory, output_filename):
    """
    Concatenates all files in a specified directory into a single file, ordered alphabetically by filename.

    Args:
    directory (str): The directory where the input files are located.
    output_filename (str): The name of the output file to create.
    """
    # Create the full path for the output file
    output_file_path = os.path.join(directory, output_filename)

    # Get a list of all files in the directory, sorted alphabetically
    files = sorted([f for f in os.listdir(directory) if f.endswith("_headless_fixed.log")])

    # Open the output file
    with open(output_file_path, 'w') as outfile:
        for filename in files:
            # Create the full path to the file
            file_path = os.path.join(directory, filename)

            # Open and read the content of the file
            with open(file_path, 'r') as infile:
                content = infile.read().strip()  # Remove any extra whitespace at the end

                # Write the file content to the output file
                outfile.write(content + '\n')  # Ensure it ends with a newline


for gain in tqdm(gains): 
    directory = DATA_ROOT / "traffic-identification" / "NR" / "SA" / str(gain) / "headless_fixed"
    output_filename = f"gnb_{gain}.log"
    concatenate_files_in_directory(directory, output_filename)

# 3. Segment by traffic types

In [ ]:
def convert_to_datetime(timestamp, fmt="%H:%M:%S.%f"):
    return datetime.datetime.strptime(timestamp, fmt)


def is_within_range(timestamp, start, end):
    start_dt = convert_to_datetime(start + ".000")
    end_dt = convert_to_datetime(end + ".999")
    timestamp_dt = convert_to_datetime(timestamp)
    return start_dt <= timestamp_dt <= end_dt


def save_lines_to_file(lines, label, output_folder):
    """Save the given lines to a file determined by the label."""
    output_path = os.path.join(output_folder, f"{label}.log")
    with open(output_path, 'w') as file:
        file.writelines(lines)


def save_log_segment(input_file_path, segment_range, label, output_folder, start_idx=0):
    timestamp_regex = re.compile(r"(\d{2}:\d{2}:\d{2}\.\d{3})")
    selected_lines = []
    started = False
    ended = False

    with open(input_file_path, 'r') as input_file:
        lines = input_file.readlines()
        t = tqdm(range(start_idx, len(lines)))
        t.set_postfix({"label": label})
        for line_idx in t:
            line = lines[line_idx]
            timestamp_match = timestamp_regex.search(line)  # Using search instead of match
            if timestamp_match:
                timestamp = timestamp_match.group(1)
                if is_within_range(timestamp, *segment_range):
                    if not started:
                        started = True
                else:
                    if started:
                        ended = True

            if started and not ended:
                selected_lines.append(line)
            elif ended:
                save_lines_to_file(selected_lines, label, output_folder)
                return line_idx
        save_lines_to_file(selected_lines, label, output_folder)
        return line_idx
    
    
def process_and_save_log_segments(input_file_path, segments, output_folder):
    """Process log file and save segments into labeled files."""
    start_idx = 0
    for segment_range, label in segments.items():
        start_idx = save_log_segment(input_file_path, segment_range, label, output_folder, start_idx)


for gain in gains: 
    print(gain)
    input_file_path = DATA_ROOT / "traffic-identification" / "NR" / "SA" / str(gain) / "headless_fixed" / f"gnb_{gain}.log"
    output_folder = DATA_ROOT / "traffic-identification" / "NR" / "SA" / str(gain) / "segments"
    segments = all_segments[gain]
    process_and_save_log_segments(input_file_path, segments, output_folder)

# 4. Form pkl files

In [ ]:
def find_timestamp_extremes(file_path):
    # This regex assumes your timestamps are formatted like "13:20:45.574"
    timestamp_regex = re.compile(r'(\d{2}:\d{2}:\d{2}\.\d{3})')
    min_time = None
    max_time = None

    with open(file_path, 'r') as file:
        for line in file:
            match = timestamp_regex.search(line)
            if match:
                # Convert the found timestamp to a datetime object
                current_time = datetime.datetime.strptime(match.group(1), '%H:%M:%S.%f').time()
                if min_time is None or current_time < min_time:
                    min_time = current_time
                if max_time is None or current_time > max_time:
                    max_time = current_time

    return min_time, max_time


for gain in gains:
    labels = list(all_segments[gain].values())
    for child_label in labels:
        # Step 1 : put the path of the child_log and get its time stamps (alternately use segments keys as an argument for timetable)
        child_file_path = DATA_ROOT / "traffic-identification" / "NR" / "SA" / str(gain) / "segments" / f"{child_label}.log"
        start_time, end_time = find_timestamp_extremes(child_file_path)

        # Step 2 : preprocess the child_log
        logfile = AmariNSALogFile(
            read_path= child_file_path,
            timetable=[((start_time, end_time), child_label)],
            window_size=1,
            pca_n_components=4,
            tb_len_threshold=300)

        # STEP 3 : convert to .pkl each child_log
        output_file_path = DATA_ROOT / "traffic-identification" / "NR" / "SA" / str(gain) / "pkl" / f"{child_label}_{gain}.pkl"
        with open(output_file_path, "wb") as file:
            pickle.dump(logfile, file)
        print(f'Successfully saved {child_label}_{gain}.pkl')

# 5. Gather column metadata

In [ ]:
hybrid_encoder = SrsRANLteHybridEncoder()
pkl_file_paths = utils.listdir_with_suffix(DATA_ROOT / "traffic-identification" / "NR" / "SA" / "pkl", ".pkl")
hybrid_encoder.collect_columns_metadata(pkl_file_paths)
hybrid_encoder.save_columns_metadata(DATA_ROOT / "traffic-identification" / "NR" / "SA" / "pkl" / "columns_metadata.json")

# 6. Fit MinMaxScalers and OneHotEncoders

In [ ]:
hybrid_encoder.fit(pkl_file_paths)
# save
with open(DATA_ROOT / "traffic-identification" / "NR" / "SA" / "pkl" / "hybrid_encoder.pickle", "wb") as file:
    pickle.dump(hybrid_encoder, file)

# 7. Transform pkl files

In [ ]:
hybrid_encoder.transform(pkl_file_paths)